# YOLO 物件偵測訓練

In [10]:
PRETRAIN_MODEL = "yolo11n.pt"
LABEL_DIR = "/media/jianhua/HDD 1T/mot_annotations"
DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
YOLO_LABEL_DIR = "/media/jianhua/HDD 1T/yolo_obj_detect"

## 轉換格式 COCO -> YOLO

In [11]:
from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=LABEL_DIR,
    save_dir=YOLO_LABEL_DIR
)

Annotations /media/jianhua/HDD 1T/mot_annotations/train.json: 100% ━━━━━━━━━━━━ 27226/27226 8.3Kit/s 3.3s0.1ss
Annotations /media/jianhua/HDD 1T/mot_annotations/val.json: 100% ━━━━━━━━━━━━ 8579/8579 8.6Kit/s 1.0s0.1s
COCO data converted successfully.
Results saved to /media/jianhua/HDD 1T/yolo_obj_detect


In [12]:
import os
import shutil

src_labels = os.path.join(YOLO_LABEL_DIR, "labels")
dst_labels = os.path.join(DATASET_DIR, "labels")

# 1️⃣ 如果 DATASET_DIR 已有 labels 就刪掉
if os.path.exists(dst_labels):
    shutil.rmtree(dst_labels)

# 2️⃣ 搬移 labels 資料夾
shutil.move(src_labels, dst_labels)

# 3️⃣ 刪掉整個 YOLO_LABEL_DIR
if os.path.exists(YOLO_LABEL_DIR):
    shutil.rmtree(YOLO_LABEL_DIR)

print("labels moved and YOLO_LABEL_DIR removed.")

# ================= 新增的部分 =================

# 定義 classes.txt 的來源與目標路徑
src_classes = "classes.txt" # 當下目錄下的 classes.txt
dst_train_dir = os.path.join(DATASET_DIR, "labels", "train")
dst_val_dir = os.path.join(DATASET_DIR, "labels", "val")

# 確保目標資料夾 train 和 val 存在 (避免 shutil.copy 找不到資料夾報錯)
os.makedirs(dst_train_dir, exist_ok=True)
os.makedirs(dst_val_dir, exist_ok=True)

# 4️⃣ 複製 classes.txt 到 train 與 val 資料夾
if os.path.exists(src_classes):
    shutil.copy(src_classes, dst_train_dir)
    shutil.copy(src_classes, dst_val_dir)
    print("classes.txt successfully copied to train and val directories.")
else:
    print(f"⚠️ 警告：在當下目錄找不到 {src_classes}，略過複製步驟。")

labels moved and YOLO_LABEL_DIR removed.
classes.txt successfully copied to train and val directories.


## 開始訓練

In [13]:
from ultralytics import YOLO
import os

model = YOLO(PRETRAIN_MODEL)  # load a pretrained model (recommended for training)

val_classes = os.path.join(DATASET_DIR, "labels/val/classes.txt")
train_classes = os.path.join(DATASET_DIR, "labels/train/classes.txt")

if os.path.exists(train_classes) and os.path.exists(val_classes):
    results = model.train(data="data.yaml", epochs=10, imgsz=640, batch=32)
else:
    raise FileNotFoundError(f"Missing required files or directories: {val_classes} {train_classes}")

New https://pypi.org/project/ultralytics/8.4.23 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.9.25 torch-2.0.0 CUDA:0 (NVIDIA GeForce RTX 4070, 11866MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False,